# 🎆 Convolutions

In this notebook, we'll walk through how convolutional filters can pick out different aspects of an image

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from skimage import data
from skimage.color import rgb2gray
from skimage.transform import resize

import torch
from torch import nn
import lightning as L

## 0. Original Input Image <a name="orig"></a>

In [ ]:
im = rgb2gray(data.coffee())
im = resize(im, (64, 64))
print(im.shape)

# PyTorch convolutions expect a batch of images shaped (N, C, H, W):
# here 1 image with 1 (grey) channel
im_tensor = torch.tensor(im, dtype=torch.float32)[None, None]
print(im_tensor.shape)

plt.axis("off")
plt.imshow(im, cmap="gray")

## Horizontal Edge Filter <a name="hor"></a>

In [ ]:
class FixedFilter(L.LightningModule):
    """A single 3x3 convolution whose weights we set by hand instead of learning them.

    A LightningModule is an nn.Module plus training hooks. There is nothing to
    train in this notebook, so we only use its forward pass (no Trainer needed).
    """

    def __init__(self, kernel, stride=1):
        super().__init__()
        # padding=1 adds a border of zeros so a stride-1 output keeps the input size
        self.conv = nn.Conv2d(1, 1, kernel_size=3, stride=stride, padding=1, bias=False)
        with torch.no_grad():
            # Conv2d weights are shaped (out_channels, in_channels, kH, kW) = (1, 1, 3, 3)
            self.conv.weight.copy_(torch.tensor(kernel, dtype=torch.float32)[None, None])

    def forward(self, x):
        return self.conv(x)


filter1 = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])

# At every position the layer multiplies the 3x3 patch by the filter
# element-wise and sums the result
with torch.no_grad():
    new_image = FixedFilter(filter1)(im_tensor)[0, 0]  # (1, 1, H, W) -> (H, W)

plt.axis("off")
plt.imshow(new_image.numpy(), cmap="Greys")

## Vertical Edge Filter <a name="ver"></a>

In [ ]:
filter2 = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])

with torch.no_grad():
    new_image = FixedFilter(filter2)(im_tensor)[0, 0]

plt.axis("off")
plt.imshow(new_image.numpy(), cmap="Greys")

## Horizontal Edge Filter with Stride 2 <a name="hor2"></a>

In [ ]:
filter1 = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])

stride = 2

# With stride=2 the filter jumps 2 pixels at a time, halving the output size
with torch.no_grad():
    new_image = FixedFilter(filter1, stride=stride)(im_tensor)[0, 0]
print(new_image.shape)

plt.axis("off")
plt.imshow(new_image.numpy(), cmap="Greys")

## Vertical Edge Filter with Stride 2 <a name="ver2"></a>

In [ ]:
filter2 = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])

stride = 2

with torch.no_grad():
    new_image = FixedFilter(filter2, stride=stride)(im_tensor)[0, 0]
print(new_image.shape)

plt.axis("off")
plt.imshow(new_image.numpy(), cmap="Greys")